# Bluebikes New Stations

For a given `YYYY-MM` month (defaults to the latest complete calendar month), finds stations that appear as a `start_station_name` in that month's trip data but not in the previous month's, and plots them on a Folium map using `start_lat`/`start_lng`. Each marker's popup shows how many days in the month the station was used, its average daily ridership, and its rank by total ridership among all stations that month.

Pulls monthly trip-history data from the [Bluebikes/Hubway System Data bucket](https://s3.amazonaws.com/hubway-data/index.html). Files are named `{YYYYMM}-bluebikes-tripdata.zip` and are available from `201805` onward.

In [ ]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

import folium
import pandas as pd
import requests
import yaml

from shared.plots.style import PALETTE

## Params

In [ ]:
MONTH = None  # YYYY-MM, or None for the latest complete calendar month

## Load or fetch trip data

Fetches the target month plus the month before it, so new stations can be identified by set difference. Each month is cached independently at `data/raw/bluebikes_tripdata/{yyyymm}/data.csv`, so re-running the notebook only fetches months not already on disk.

In [ ]:
BASE_URL = "https://s3.amazonaws.com/hubway-data"


def latest_complete_month():
    first_of_this_month = pd.Timestamp.today().normalize().replace(day=1)
    return (first_of_this_month - pd.DateOffset(months=1)).strftime("%Y-%m")


def fetch_month_tripdata(yyyymm):
    raw_path = Path(f"data/raw/bluebikes_tripdata/{yyyymm}/data.csv")

    if not raw_path.exists():
        # Almost every month is "{yyyymm}-bluebikes-tripdata.zip"; a rare
        # month (e.g. 202511) ships as "{yyyymm}-bluebikes-tripdata.csv.zip" instead.
        for suffix in ("-bluebikes-tripdata.zip", "-bluebikes-tripdata.csv.zip"):
            url = f"{BASE_URL}/{yyyymm}{suffix}"
            response = requests.get(url, timeout=60)
            if response.ok:
                break
        response.raise_for_status()

        with ZipFile(BytesIO(response.content)) as zf:
            csv_name = next(
                name
                for name in zf.namelist()
                if name.endswith(".csv") and "__MACOSX" not in name
            )
            raw_path.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(csv_name) as src, open(raw_path, "wb") as dst:
                dst.write(src.read())

        print(f"Fetched and cached: {raw_path}")
    else:
        print(f"Loaded cached pull: {raw_path}")

    return raw_path


month = MONTH or latest_complete_month()
prior_month = (pd.Period(month, freq="M") - 1).strftime("%Y-%m")

month_path = fetch_month_tripdata(pd.Period(month, freq="M").strftime("%Y%m"))
prior_month_path = fetch_month_tripdata(pd.Period(prior_month, freq="M").strftime("%Y%m"))

trips = pd.read_csv(
    month_path,
    usecols=["started_at", "start_station_name", "start_lat", "start_lng"],
)
prior_station_names = pd.read_csv(prior_month_path, usecols=["start_station_name"])[
    "start_station_name"
].unique()

trips.head()

## Per-station stats and new-station diff

"Days used" is the count of distinct calendar days with at least one trip starting at the station; "daily ridership" is total trips at the station divided by days used (its average ridership on a day it saw any activity); rank is by total trips among all stations that started a trip in the month.

In [ ]:
trips["trip_date"] = trips["started_at"].str.slice(0, 10)

station_stats = (
    trips.groupby("start_station_name")
    .agg(
        total_trips=("trip_date", "size"),
        days_used=("trip_date", "nunique"),
        lat=("start_lat", "median"),
        lng=("start_lng", "median"),
    )
    .reset_index()
)
station_stats["daily_ridership"] = station_stats["total_trips"] / station_stats["days_used"]
station_stats["rank"] = station_stats["total_trips"].rank(ascending=False, method="min").astype(int)
station_stats = station_stats.sort_values("rank")

new_stations = station_stats[~station_stats["start_station_name"].isin(prior_station_names)].copy()
print(f"{len(new_stations)} new station(s) in {month} vs {prior_month}")
new_stations

## Map: new stations

In [ ]:
output_dir = Path(f"outputs/{month}")
output_dir.mkdir(parents=True, exist_ok=True)


def load_carto_api_key():
    secrets_path = Path("../../../config_secrets.yml")
    if not secrets_path.exists():
        print("Warning: config_secrets.yml not found; CARTO map tiles may be rate-limited. See config_secrets_template.yml.")
        return None
    secrets = yaml.safe_load(secrets_path.read_text()) or {}
    key = secrets.get("api_keys", {}).get("carto")
    if not key:
        print("Warning: no CARTO api key set in config_secrets.yml; map tiles may be rate-limited.")
    return key


if new_stations.empty:
    print(f"No new stations in {month} vs {prior_month}; skipping map.")
else:
    carto_api_key = load_carto_api_key()
    tiles = "https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png"
    if carto_api_key:
        tiles += f"?key={carto_api_key}"

    map_center = [new_stations["lat"].mean(), new_stations["lng"].mean()]
    station_map = folium.Map(
        location=map_center, zoom_start=13, tiles=tiles, attr="CARTO"
    )

    total_stations = len(station_stats)
    for row in new_stations.itertuples():
        popup_html = (
            f"<b>{row.start_station_name}</b><br>"
            f"Days used in {month}: {row.days_used}<br>"
            f"Daily ridership: {row.daily_ridership:.1f} trips/day<br>"
            f"Total trips: {row.total_trips:,}<br>"
            f"Rank: {row.rank:,} of {total_stations:,} stations"
        )
        folium.CircleMarker(
            location=[row.lat, row.lng],
            radius=8,
            color=PALETTE[0],
            fill=True,
            fill_color=PALETTE[0],
            fill_opacity=0.85,
            weight=2,
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=row.start_station_name,
        ).add_to(station_map)

    map_path = output_dir / "new_stations_map.html"
    station_map.save(map_path)
    print(f"Saved map: {map_path}")
    display(station_map)